## Opis podejścia
Nagrania przepisujemy whisperem: large-v3-turbo -> następnie poprawiamy w tekście keywordsy IT za pomocą bielika 4.5B, uruchomionego lokalnie przez Ollame. Bielik dostaje cały tekst i listę nazw ze słownika z poleceniem, aby poprawił tylko błędnie zapisane nazwy.

Pojawił się na tym etapie małuy problem bo bielik zmieniał tez inne słowa i tekst wychodził znacznie gorszy niz po samym whisperze.

Tutaj pojawił się straznik, który porównuje tekst przed korekta i po niej słowo po słowie i zostawia tylko te zmiany, które wporwadzają nową nazwe ze słownika. Pozostałe zmiany oraz dopisane słowa wracają do wersji Whispera

Jakośc mierzymy na dwóch poziomach. Na Poziomie terminów liczymy precision, recall i F1, wyszukujac nazwy ze slownika z uwzględnieniem odmiany. Na poziomie całego tekstu liczymy CER i WER. Nagrania fałszywe nie wchodzą do P/R/F1, sprawdzamy tylko liczbę fałszywych alarmów


## Ładowanie whispera

In [ ]:
from faster_whisper import WhisperModel

model_size = "large-v3-turbo"
ASR_NAME = model_size

model = WhisperModel(
    model_size,
    device="cpu",
    compute_type="int8",
)


## Transkrypcja nagrań

In [ ]:
import dataclasses
import json
import time
from pathlib import Path

import pandas as pd


DATA = Path("..") / "data"
OUT = DATA / "asr" / ASR_NAME
OUT.mkdir(parents=True, exist_ok=True)


for file_id in pd.read_csv(DATA / "recordings.csv")["file_id"]:
    audio_file = DATA / "audio" / f"{file_id}.wav"
    out_file = OUT / f"{file_id}.json"

    if out_file.exists():
        print(f"Skipping {file_id}: output already exists.")
        continue
    if not audio_file.exists():
        print(f"Skipping {file_id}: no audio file.")
        continue

    start_time = time.time()
    segments, info = model.transcribe(
        str(audio_file),
        beam_size=5,
        language="pl",
        word_timestamps=True,
    )
    segments = list(segments)
    text = " ".join(s.text.strip() for s in segments)

    result = {"text": text, "segments": [dataclasses.asdict(s) for s in segments]}
    tmp_file = out_file.with_suffix(".tmp")
    tmp_file.write_text(
        json.dumps(result, ensure_ascii=False, indent=1), encoding="utf-8"
    )
    tmp_file.replace(out_file)

    print(f"Transcribed {file_id} in {time.time() - start_time:.2f} seconds.")


Skipping rec_201: output already exists.
Skipping rec_202: output already exists.
Skipping rec_203: output already exists.
Skipping rec_204: output already exists.
Skipping rec_205: output already exists.
Skipping rec_101: output already exists.
Skipping rec_102: output already exists.
Skipping rec_103: output already exists.
Skipping rec_104: output already exists.
Skipping rec_105: output already exists.
Skipping rec_001: output already exists.
Skipping rec_002: output already exists.
Skipping rec_003: output already exists.
Skipping rec_004: output already exists.
Skipping rec_005: output already exists.


Słownik terminów

In [84]:
import csv
import re
from pathlib import Path

DATA = Path("..") / "data"

PUNCT = re.compile(r"[,;:!?…\"'„”()\[\]{}]")


def tokenize(text):
    text = PUNCT.sub(" ", text.lower())
    tokens = [t.strip(".-") for t in text.split()]
    return [t for t in tokens if t]


FORMS = {}
conflicts = []
with open(DATA / "terms_it.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        term = row["term"]
        for form in [term] + row["aliases"].split("|"):
            tokens = tuple(tokenize(form))
            if not tokens:
                continue
            if tokens in FORMS and FORMS[tokens] != term:
                conflicts.append((tokens, FORMS[tokens], term))
            FORMS.setdefault(tokens, term)

MAX_LEN = max(len(k) for k in FORMS)
print(len(FORMS), "form,", MAX_LEN, "tokenów w najdłuższej")
print(
    FORMS[("vercel",)],
    "|",
    FORMS[("merge", "request")],
    "|",
    FORMS[("net",)],
    "|",
    FORMS[("c#",)],
)
print("konflikty:", conflicts)


589 form, 4 tokenów w najdłuższej
Vercel | merge request | .NET | C#
konflikty: []


## Wyszukiwanie terminów w tekście

In [85]:
MAX_ENDING = 4
MIN_STEM = 4


def match_at(tokens, i):
    for n in range(MAX_LEN, 0, -1):
        span = tuple(tokens[i : i + n])
        if len(span) < n:
            continue
        head, last = span[:-1], span[-1]
        if span in FORMS:
            return FORMS[span], n
        if "-" in last:
            beforehyphen = head + (last.split("-")[0],)
            if beforehyphen in FORMS:
                return FORMS[beforehyphen], n

        for k in range(1, MAX_ENDING + 1):
            stem = last[:-k]
            if len(stem) >= MIN_STEM and head + (stem,) in FORMS:
                return FORMS[head + (stem,)], n
    return None


def find_terms(text):
    tokens = tokenize(text)
    found = []
    i = 0
    while i < len(tokens):
        match = match_at(tokens, i)
        if match:
            term, n = match
            found.append(term)
            i += n
        else:
            i += 1
    return found


tests = [
    "Frontend stoi na Vercelu, a backend na Railwayu.",
    "Robimy merge request i code review w GitLabie.",
    "Używamy LLM-ów i RAG-owego podejścia na AWS-ie.",
    "Ta gitara jest fajna, a to jest AI.",
]
for t in tests:
    print(find_terms(t), "<-", t)


['frontend', 'Vercel', 'backend', 'Railway'] <- Frontend stoi na Vercelu, a backend na Railwayu.
['merge request', 'code review', 'GitLab'] <- Robimy merge request i code review w GitLabie.
['LLM', 'RAG', 'AWS'] <- Używamy LLM-ów i RAG-owego podejścia na AWS-ie.
['AI'] <- Ta gitara jest fajna, a to jest AI.


## Kontrola wyszukiwania na tekście

In [86]:
import re
from collections import Counter

MARK = re.compile(r"\{([^{}|]*)\|([^{}|]*)\}")

tp = fp = fn = 0
for file_id in pd.read_csv(DATA / "recordings.csv")["file_id"]:
    gold = (DATA / "transcripts" / f"{file_id}.txt").read_text(encoding="utf-8")

    gold_terms = Counter(canon for _, canon in MARK.findall(gold))
    plain = MARK.sub(lambda m: m.group(1), gold)
    found_terms = Counter(find_terms(plain))

    hit = sum((gold_terms & found_terms).values())
    extra = found_terms - gold_terms
    missed = gold_terms - found_terms

    tp += hit
    fp += sum(extra.values())
    fn += sum(missed.values())

    print(file_id, "gold:", sum(gold_terms.values()), "trafione:", hit)
    if extra:
        print("   nadmiarowe:", extra)
    if missed:
        print("   brakujące:", missed)

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")

rec_201 gold: 7 trafione: 6
   brakujące: Counter({'rollback': 1})
rec_202 gold: 4 trafione: 4
rec_203 gold: 5 trafione: 5
   nadmiarowe: Counter({'POC': 1})
rec_204 gold: 7 trafione: 7
rec_205 gold: 0 trafione: 0
   nadmiarowe: Counter({'Docker': 1, 'commit': 1, 'POC': 1})
rec_101 gold: 8 trafione: 8
rec_102 gold: 11 trafione: 11
rec_103 gold: 15 trafione: 15
rec_104 gold: 15 trafione: 15
rec_105 gold: 0 trafione: 0
   nadmiarowe: Counter({'branch': 1, 'REST': 1})
rec_001 gold: 19 trafione: 19
   nadmiarowe: Counter({'backend': 1})
rec_002 gold: 17 trafione: 17
   nadmiarowe: Counter({'develop': 1})
rec_003 gold: 18 trafione: 17
   brakujące: Counter({'support': 1})
rec_004 gold: 13 trafione: 13
rec_005 gold: 0 trafione: 0
   nadmiarowe: Counter({'sprint': 3})
Precision: 0.926, Recall: 0.986, F1: 0.955


## Wynik bazowy whispera

In [87]:
import json

recordings = pd.read_csv(DATA / "recordings.csv")


def gold_terms(file_id):
    gold = (DATA / "transcripts" / f"{file_id}.txt").read_text(encoding="utf-8")
    return Counter(canon for _, canon in MARK.findall(gold))


def whisper_text(file_id):
    path = DATA / "asr" / ASR_NAME / f"{file_id}.json"
    return json.loads(path.read_text(encoding="utf-8"))["text"]


def score(get_text, title):
    print(title)
    tp = fp = fn = neg_fp = 0
    for row in recordings.itertuples():
        gold = gold_terms(row.file_id)
        found = Counter(find_terms(get_text(row.file_id)))
        extra = found - gold
        missed = gold - found

        if row.sample_type == "negative":
            neg_fp += sum(extra.values())
            print(f"{row.file_id} (negatywne): nadmiarowe {sum(extra.values())}")
            continue

        hit = sum((gold & found).values())
        tp += hit
        fp += sum(extra.values())
        fn += sum(missed.values())
        print(
            f"{row.file_id}: gold {sum(gold.values())}, trafione {hit}, "
            f"nadmiarowe {sum(extra.values())}, pominięte {sum(missed.values())}"
        )
    p = tp / (tp + fp)
    r = tp / (tp + fn)
    f1 = 2 * p * r / (p + r)
    print(
        f"P={p:.3f}, R={r:.3f}, F1={f1:.3f}  |  fałszywe alarmy w negatywnych: {neg_fp}"
    )


score(whisper_text, "SUROWY WHISPER")

SUROWY WHISPER
rec_201: gold 7, trafione 5, nadmiarowe 0, pominięte 2
rec_202: gold 4, trafione 4, nadmiarowe 0, pominięte 0
rec_203: gold 5, trafione 5, nadmiarowe 1, pominięte 0
rec_204: gold 7, trafione 7, nadmiarowe 2, pominięte 0
rec_205 (negatywne): nadmiarowe 3
rec_101: gold 8, trafione 6, nadmiarowe 0, pominięte 2
rec_102: gold 11, trafione 10, nadmiarowe 0, pominięte 1
rec_103: gold 15, trafione 7, nadmiarowe 0, pominięte 8
rec_104: gold 15, trafione 8, nadmiarowe 0, pominięte 7
rec_105 (negatywne): nadmiarowe 0
rec_001: gold 19, trafione 6, nadmiarowe 0, pominięte 13
rec_002: gold 17, trafione 15, nadmiarowe 1, pominięte 2
rec_003: gold 18, trafione 15, nadmiarowe 2, pominięte 3
rec_004: gold 13, trafione 3, nadmiarowe 0, pominięte 10
rec_005 (negatywne): nadmiarowe 2
P=0.938, R=0.655, F1=0.771  |  fałszywe alarmy w negatywnych: 5


## Korekta bielikiem

In [88]:
import ollama

BIELIK = "hf.co/speakleash/Bielik-11B-v3.0-Instruct-GGUF:Q4_K_M"
TERMS = sorted(set(FORMS.values()))

SYSTEM = (
    "Jesteś korektorem transkrypcji mowy. Dostajesz tekst z rozpoznawania mowy "
    "oraz listę poprawnych nazw technicznych. Popraw TYLKO te słowa, które są "
    "błędnie zapisanymi nazwami z listy. Zachowaj polską odmianę. Nie zmieniaj "
    "żadnych innych słów, nie dodawaj ani nie usuwaj niczego. "
    "Odpowiedz wyłącznie poprawionym tekstem."
    "np. Bercelu->Vercel"
)


def correct(text):
    user = "Lista nazw: " + ", ".join(TERMS) + "\n\nTekst:\n" + text
    response = ollama.chat(
        model=BIELIK,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "system", "content": user},
        ],
        options={"temperature": 0},
    )
    return response["message"]["content"].strip()


original = whisper_text("rec_104")
fixed = correct(original)
print("WHISPER:\n" + original)
print("\nBIELIK:\n" + fixed)

WHISPER:
Głównie programuję w C-Sharpie i dotnecie bo od niego zaczynałem na tym stoi backend vocary nauczyłem się go już jakiś czas temu więc znam ekosystem ASP.NET Core Entity Framework dobrze mi się pracuje w tym ekosystemie drugim językiem jest u mnie Python używam go do skryptów do narzędzia analitycznego które planuję i w pracy mi się czasem przydaje jest szybki do prototypowania ma sporo bibliotek do danych czy tam też tytułu. Czytam też tytułu. Czytam też tytułu. Czytam też tytułu. Czytam też Typescript bo frontend jest u nas w Typeskrypcie robię przeglady PR-u ale sam raczej nic w nim nie piszę. Javy nie znam dobrze kojarzę głównie z dużymi firmami z korporacjami z większymi systemami. Na UDSJ jest widziałem tylko przy okazji frontendu generalnie staram się wybrać narzędzia do zadania ale dotny jest dla mnie tym w którym czuję się Jakby w tych momentu. Czytam też tytułu. Czytam też tytułem w tytułem w tym wieszczu i tułem w bytom nie?

BIELIK:
Głównie programuję w C# i .NET, b

## Stranik zmian

In [89]:
import difflib


def guard(original, corrected):
    a = original.split()
    b = corrected.split()
    result = []
    log = []
    matcher = difflib.SequenceMatcher(None, a, b, autojunk=False)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        old = " ".join(a[i1:i2])
        new = " ".join(b[j1:j2])
        if tag == "equal":
            result += a[i1:i2]
            continue
        gained = Counter(find_terms(new)) - Counter(find_terms(old))
        if gained and tag == "replace":
            result += b[j1:j2]
            log.append(("PRZYJĘTE", old, new))
        else:
            result += a[i1:i2]
            log.append(("COFNIĘTE", old, new))
    return " ".join(result), log


original = whisper_text("rec_102")
fixed = correct(original)
guarded, log = guard(original, fixed)
for tag, old, new in log:
    print(f"{tag} | {old} -> {new}")

## Korekta wszystkich nagrań

In [90]:
BIELIK_OUT = DATA / "asr" / f"{ASR_NAME}-bielik-11b"
BIELIK_OUT.mkdir(parents=True, exist_ok=True)

for file_id in recordings["file_id"]:
    out_file = BIELIK_OUT / f"{file_id}.json"
    if out_file.exists():
        print(f"Skipping {file_id}: output already exists.")
        continue

    original = whisper_text(file_id)
    raw = correct(original).replace("*", "")
    guarded, log = guard(original, raw)

    result = {"text": guarded, "raw": raw, "log": log}
    out_file.write_text(
        json.dumps(result, ensure_ascii=False, indent=1), encoding="utf-8"
    )

    accepted = [(old, new) for tag, old, new in log if tag == "PRZYJĘTE"]
    print(file_id, "przyjęte:", accepted)


def guarded_text(file_id):
    path = BIELIK_OUT / f"{file_id}.json"
    return json.loads(path.read_text(encoding="utf-8"))["text"]


def raw_bielik_text(file_id):
    path = BIELIK_OUT / f"{file_id}.json"
    return json.loads(path.read_text(encoding="utf-8"))["raw"]


score(whisper_text, "SUROWY WHISPER")
score(raw_bielik_text, "BIELIK BEZ STRAŻNIKA")
score(guarded_text, "BIELIK + STRAŻNIK")

rec_201 przyjęte: []
rec_202 przyjęte: []
rec_203 przyjęte: []
rec_204 przyjęte: []
rec_205 przyjęte: []
rec_101 przyjęte: [('CI-CD.', 'CI/CD.'), ('kubernete się wdrożenie to', 'Kubernetesie,')]
rec_102 przyjęte: []
rec_103 przyjęte: []
rec_104 przyjęte: [('C-Sharpie', 'C#'), ('vocary nauczyłem', 'Vocare. Nauczyłem'), ('Typeskrypcie', 'TypeScript -'), ('przeglady PR-u', 'przeglądy PR-ów,'), ('dotny', '.NET')]
rec_105 przyjęte: []
rec_001 przyjęte: [('LLMów', 'LLM-ów,'), ("Hackfire'a", "Hangfire'a,")]
rec_002 przyjęte: []
rec_003 przyjęte: []
rec_004 przyjęte: []
rec_005 przyjęte: []
SUROWY WHISPER
rec_201: gold 7, trafione 5, nadmiarowe 0, pominięte 2
rec_202: gold 4, trafione 4, nadmiarowe 0, pominięte 0
rec_203: gold 5, trafione 5, nadmiarowe 1, pominięte 0
rec_204: gold 7, trafione 7, nadmiarowe 2, pominięte 0
rec_205 (negatywne): nadmiarowe 3
rec_101: gold 8, trafione 6, nadmiarowe 0, pominięte 2
rec_102: gold 11, trafione 10, nadmiarowe 0, pominięte 1
rec_103: gold 15, trafione 7,

## Porównanie wyników dev i test

In [91]:
import jiwer


def gold_plain(file_id):
    gold = (DATA / "transcripts" / f"{file_id}.txt").read_text(encoding="utf-8")
    return " ".join(tokenize(MARK.sub(lambda m: m.group(1), gold)))


def summary(get_text, name, split):
    rows = recordings[recordings["split"] == split]
    tp = fp = fn = neg_fp = 0
    refs = []
    hyps = []
    for row in rows.itertuples():
        text = get_text(row.file_id)
        refs.append(gold_plain(row.file_id))
        hyps.append(" ".join(tokenize(text)))

        gold = gold_terms(row.file_id)
        found = Counter(find_terms(text))
        extra = found - gold
        if row.sample_type == "negative":
            neg_fp += sum(extra.values())
            continue
        tp += sum((gold & found).values())
        fp += sum(extra.values())
        fn += sum((gold - found).values())

    p = tp / (tp + fp)
    r = tp / (tp + fn)
    return {
        "zbiór": split,
        "wariant": name,
        "P": round(p, 3),
        "R": round(r, 3),
        "F1": round(2 * p * r / (p + r), 3),
        "fałszywe w neg.": neg_fp,
        "CER": round(jiwer.cer(refs, hyps), 3),
        "WER": round(jiwer.wer(refs, hyps), 3),
    }


variants = [
    (whisper_text, "Whisper"),
    (raw_bielik_text, "Bielik bez strażnika"),
    (guarded_text, "Bielik + strażnik"),
]
table = pd.DataFrame(
    [summary(f, name, split) for split in ("dev", "test") for f, name in variants]
)
table


,zbiór,wariant,P,R,F1,fałszywe w neg.,CER,WER
0,dev,Whisper,0.936,0.611,0.739,2,0.271,0.350
1,dev,Bielik bez strażnika,0.959,0.653,0.777,2,0.256,0.329
2,dev,Bielik + strażnik,0.941,0.667,0.780,2,0.269,0.345
3,test,Whisper,0.940,0.701,0.803,3,0.279,0.375
4,test,Bielik bez strażnika,0.963,0.776,0.860,3,0.347,0.423
5,test,Bielik + strażnik,0.945,0.776,0.852,3,0.278,0.371


In [92]:
file_id = "rec_001"
result = json.loads((BIELIK_OUT / f"{file_id}.json").read_text(encoding="utf-8"))

print("WHISPER:\n" + whisper_text(file_id))
print("\nPO KOREKCIE:\n" + result["text"])
print("\nZMIANY:")
for tag, old, new in result["log"]:
    print(f"{tag} | {old} -> {new}")


WHISPER:
Jeśli chodzi o projekt z którego jestem najbardziej dumny to jest to platforma Vocare która jest wdrożona produkcyjnie i można z niej korzystać mamy już nawet stałych użytkowników i codziennie rejestrujemy ruch na platformie jest to aplikacja do doradztwa zawodowego wspieranego technologiami AI w której korzystamy z dużych LLMów ale planujemy rozwój w postaci własnego systemu zaawansowanego systemu nr. systemu ważха Reagowego z Harekowcą marriagem Mirror priority wektory do wektoryzacji używamy pluginu pgvector Zadania w tle delegujemy za pomocą Hackfire'a a frontend oczywiście stoi na Wercelu

PO KOREKCIE:
Jeśli chodzi o projekt z którego jestem najbardziej dumny to jest to platforma Vocare która jest wdrożona produkcyjnie i można z niej korzystać mamy już nawet stałych użytkowników i codziennie rejestrujemy ruch na platformie jest to aplikacja do doradztwa zawodowego wspieranego technologiami AI w której korzystamy z dużych LLM-ów, ale planujemy rozwój w postaci własnego sys